AI-Powered Cybersecurity Threat Reporter
1. Project Overview
This project implements a Local RAG (Retrieval-Augmented Generation) system designed to analyze security logs and technical documents. It uses Ollama (Gemma) for local inference and ChromaDB for vector storage.

The system follows a three-step pipeline:

Ingestion: Loading and splitting security documents (PDFs, TXT, CSV).

Vectorization: Converting text into mathematical embeddings using sentence-transformers.

Retrieval & Generation: Finding relevant context in the database to help the LLM generate a structured threat report.

2. The RAG Pipeline Architecture
Explain the flow of data using this breakdown:

Document Loading: We use DirectoryLoader to scan the /data folder for threat intelligence reports and logs.

Text Splitting: To stay within the model's context window, documents are broken into chunks of 1000 characters with a 150-character overlap.

Embeddings: We use the all-MiniLM-L6-v2 model from HuggingFace to create embeddings. This model is efficient for CPU-based local processing.

Vector Store: ChromaDB acts as our long-term memory, storing these embeddings for fast similarity searches.

3. AI Constraints and Role-Play
To ensure the system remains a specialized tool, we have implemented a Strict System Prompt. The AI is instructed to:

Act as a Professional Cybersecurity AI.

Refuse non-security tasks: It will not provide entertainment or casual conversation, ensuring the system remains "on-task" during security audits.

Standardize Reporting: Every output follows a professional format including an Executive Summary, Technical Analysis, Mitigation Steps, and a 1-10 Risk Score.

First we will install all our libraries required for this project
\

In [1]:
# Install LangChain, ChromaDB, and Embedding models
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers pypdf streamlit

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Setup Ollama (Gemma4)


Because Colab is a fresh environment, we must start the Ollama server in the background and pull the gemma model.

In [2]:
import subprocess
import time

# Start Ollama server in the background
subprocess.Popen(['ollama', 'serve'])
time.sleep(10) # Give it time to start

# Pull the model
!ollama pull gemma4

Then we will import all the lib required


In [ ]:
import os
import shutil
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate


This cell creates the database structure and defines the RAG chain logic.

In [4]:

# --- CONFIGURATION ---
DATA_PATH = "data/"
DB_PATH = "db/"

# Create dummy data folder if it doesn't exist
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)
    with open(f"{DATA_PATH}security_info.txt", "w") as f:
        f.write("Brute force attacks on Port 22 usually involve SSH credential stuffing.")

def create_vector_db():
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)

    # Loaders
    loaders = [
        DirectoryLoader(DATA_PATH, glob="*.pdf", loader_cls=PyPDFLoader),
        DirectoryLoader(DATA_PATH, glob="*.txt", loader_cls=TextLoader),
        DirectoryLoader(DATA_PATH, glob="*.csv", loader_cls=CSVLoader)
    ]

    docs = []
    for loader in loaders:
        docs.extend(loader.load())

    if not docs:
        print("⚠️ No docs found. Add files to the 'data/' folder in the sidebar.")
        return None

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = text_splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_PATH)
    return vector_db

def get_rag_chain(vector_db):
    llm = Ollama(model="gemma:2b", temperature=0.1)

    template = """
    ROLE: You are a professional Cybersecurity AI.
    STRICT RULES:
    1. ONLY answer questions about cybersecurity, network logs, or threats.
    2. If the user asks for a joke or casual talk, refuse politely.
    3. Use the context below to build your report.

    CONTEXT: {context}
    USER INPUT: {question}

    REPORT FORMAT:
    ### 🛡️ Executive Summary
    ### 🔍 Technical Analysis (Timestamps, IPs, Attack Types)
    ### 🛠️ Mitigation Steps
    ### ⚠️ Risk Score (1-10)
    """

    QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)
    return RetrievalQA.from_chain_type(
        llm,
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
    )

# Initialize DB
v_db = create_vector_db()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Streamlit doesn't run natively inside a Colab cell output, so we use localtunnel to create a public URL to view our UI.

In [ ]:
# Write the UI code to a file
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import os
# We import the functions we just defined in the notebook (via a slight hack or re-definition)
# For this script to work standalone, you'd paste the logic here or import it.
st.title("🛡️ GenAI Cybersecurity Threat Reporter")
user_query = st.text_area("Describe threat/paste logs:")
if st.button("Generate Report"):
    st.info("Analysis in progress... (In Colab, use the logic cell above for direct testing)")
    """)

# Run Streamlit in the background
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏⠙⠹⠸⠼⠴⠦⠧⠇⠏

your url is: https://legal-loops-joke.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.91.205:8501



How to Run
Upload Data: Place your .pdf or .txt logs in the data/ directory.

Initialize DB: Run the Ingestion cell to create the db/ folder.

Launch UI: Use the Streamlit cell to generate the public URL and interact with the reporter.